# Notebook 01: Data Profiling
## SaaS Expansion Analytics — RavenStack Dataset
**Objective:** Understand the five-table data model, validate quality, and define the expansion target universe.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print("Libraries loaded")

Libraries loaded


In [2]:
# Load all 5 tables
accounts      = pd.read_csv('../data/raw/ravenstack_accounts.csv')
subscriptions = pd.read_csv('../data/raw/ravenstack_subscriptions.csv')
feature_usage = pd.read_csv('../data/raw/ravenstack_feature_usage.csv')
support       = pd.read_csv('../data/raw/ravenstack_support_tickets.csv')
churn         = pd.read_csv('../data/raw/ravenstack_churn_events.csv')

# Parse dates
accounts['signup_date']         = pd.to_datetime(accounts['signup_date'])
subscriptions['start_date']     = pd.to_datetime(subscriptions['start_date'])
subscriptions['end_date']       = pd.to_datetime(subscriptions['end_date'])
feature_usage['usage_date']     = pd.to_datetime(feature_usage['usage_date'])
support['submitted_at']         = pd.to_datetime(support['submitted_at'])
support['closed_at']            = pd.to_datetime(support['closed_at'])
churn['churn_date']             = pd.to_datetime(churn['churn_date'])

print("All tables loaded and dates parsed")
print(f"\naccounts:      {accounts.shape}")
print(f"subscriptions: {subscriptions.shape}")
print(f"feature_usage: {feature_usage.shape}")
print(f"support:       {support.shape}")
print(f"churn:         {churn.shape}")

All tables loaded and dates parsed

accounts:      (500, 10)
subscriptions: (5000, 14)
feature_usage: (25000, 8)
support:       (2000, 9)
churn:         (600, 9)


## 1. Dataset Overview
Understanding the table relationships before any analysis.

In [3]:
# Verify foreign key relationships
acc_ids  = set(accounts['account_id'])
sub_ids  = set(subscriptions['subscription_id'])

print("=== Foreign Key Validation ===")
print(f"accounts:        {len(acc_ids)} unique account_ids")

sub_accs = set(subscriptions['account_id'])
print(f"subscriptions:   {len(sub_accs)} unique account_ids "
      f"({len(sub_accs - acc_ids)} orphans)")

fu_subs = set(feature_usage['subscription_id'])
print(f"feature_usage:   {len(fu_subs)} unique subscription_ids "
      f"({len(fu_subs - sub_ids)} orphans)")

sup_accs = set(support['account_id'])
print(f"support:         {len(sup_accs)} unique account_ids "
      f"({len(sup_accs - acc_ids)} orphans)")

churn_accs = set(churn['account_id'])
print(f"churn:           {len(churn_accs)} unique account_ids "
      f"({len(churn_accs - acc_ids)} orphans)")

print(f"\nSubscriptions per account (avg): "
      f"{subscriptions.groupby('account_id').size().mean():.1f}")
print(f"Usage events per subscription (avg): "
      f"{feature_usage.groupby('subscription_id').size().mean():.1f}")
print(f"Support tickets per account (avg): "
      f"{support.groupby('account_id').size().mean():.1f}")

=== Foreign Key Validation ===
accounts:        500 unique account_ids
subscriptions:   500 unique account_ids (0 orphans)
feature_usage:   4967 unique subscription_ids (0 orphans)
support:         492 unique account_ids (0 orphans)
churn:           352 unique account_ids (0 orphans)

Subscriptions per account (avg): 10.0
Usage events per subscription (avg): 5.0
Support tickets per account (avg): 4.1


## 2. Date Ranges
Understanding the temporal scope of the dataset.

In [4]:
print("=== Date Ranges ===")
print(f"accounts signup:       "
      f"{accounts['signup_date'].min().date()} → "
      f"{accounts['signup_date'].max().date()}")
print(f"subscriptions start:   "
      f"{subscriptions['start_date'].min().date()} → "
      f"{subscriptions['start_date'].max().date()}")
print(f"feature usage:         "
      f"{feature_usage['usage_date'].min().date()} → "
      f"{feature_usage['usage_date'].max().date()}")
print(f"support tickets:       "
      f"{support['submitted_at'].min().date()} → "
      f"{support['submitted_at'].max().date()}")
print(f"churn events:          "
      f"{churn['churn_date'].min().date()} → "
      f"{churn['churn_date'].max().date()}")

=== Date Ranges ===
accounts signup:       2023-01-02 → 2024-12-31
subscriptions start:   2023-01-09 → 2024-12-31
feature usage:         2023-01-01 → 2024-12-31
support tickets:       2023-01-02 → 2024-12-31
churn events:          2023-01-25 → 2024-12-31


## 3. Account Profile
Plan distribution, industry mix, and referral sources.

In [10]:
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=['Plan Tier', 'Industry', 'Referral Source']
)

# Plan tier
plan_counts = accounts['plan_tier'].value_counts()
fig.add_trace(go.Bar(
    x=plan_counts.index,
    y=plan_counts.values,
    text=plan_counts.values,
    textposition='outside',
    marker_color=['#3498db', '#e67e22', '#2ecc71'],
    showlegend=False
), row=1, col=1)

# Industry
industry_counts = accounts['industry'].value_counts()
fig.add_trace(go.Bar(
    x=industry_counts.index,
    y=industry_counts.values,
    text=industry_counts.values,
    textposition='outside',
    marker_color='#3498db',
    showlegend=False
), row=1, col=2)

# Referral source
referral_counts = accounts['referral_source'].value_counts()
fig.add_trace(go.Bar(
    x=referral_counts.index,
    y=referral_counts.values,
    text=referral_counts.values,
    textposition='outside',
    marker_color='#e67e22',
    showlegend=False
), row=1, col=3)

fig.update_layout(
    title='Account Distribution',
    height=500,
    font=dict(size=12)
)
fig.show()

## 4. Active vs Churned Accounts
Defining the expansion target universe — active, non-Enterprise accounts.

In [ ]:
# Active vs churned
total        = len(accounts)
churned      = accounts['churn_flag'].sum()
active       = total - churned
trial        = accounts['is_trial'].sum()

print("=== Account Status ===")
print(f"Total accounts:   {total}")
print(f"Active:           {active} ({active/total*100:.1f}%)")
print(f"Churned:          {churned} ({churned/total*100:.1f}%)")
print(f"On trial:         {trial} ({trial/total*100:.1f}%)")

# Reactivations in churn table
reactivations = churn['is_reactivation'].sum()
print(f"\nChurn events:     {len(churn)}")
print(f"Reactivations:    {reactivations} "
      f"({reactivations/len(churn)*100:.1f}% of churn events)")
print(f"Note: {len(churn)} churn events for {total} accounts "
      f"— reactivations explain the excess")

# Expansion universe
active_accounts = accounts[accounts['churn_flag'] == False]
expansion_universe = active_accounts[
    active_accounts['plan_tier'].isin(['Basic', 'Pro'])
]
print(f"\n=== Expansion Universe ===")
print(f"Active accounts:              {len(active_accounts)}")
print(f"Active Basic + Pro accounts:  {len(expansion_universe)}")
print(f"(Enterprise excluded — no higher tier to upgrade to)")
print(f"\nExpansion universe by plan:")
print(expansion_universe['plan_tier'].value_counts())

=== Account Status ===
Total accounts:   500
Active:           390 (78.0%)
Churned:          110 (22.0%)
On trial:         97 (19.4%)

Churn events:     600
Reactivations:    61 (10.2% of churn events)
Note: 600 churn events for 500 accounts — reactivations explain the excess

=== Expansion Universe ===
Active accounts:              390
Active Basic + Pro accounts:  270
(Enterprise excluded — no higher tier to upgrade to)

Expansion universe by plan:
plan_tier
Pro      139
Basic    131
Name: count, dtype: int64


## 5. Subscription Patterns
Understanding how accounts move through plans over time.

In [13]:
# Upgrade and downgrade rates
total_subs   = len(subscriptions)
upgrades     = subscriptions['upgrade_flag'].sum()
downgrades   = subscriptions['downgrade_flag'].sum()
active_subs  = subscriptions['end_date'].isna().sum()
churned_subs = subscriptions['churn_flag'].sum()

print("=== Subscription Summary ===")
print(f"Total subscriptions:  {total_subs}")
print(f"Active (no end_date): {active_subs} "
      f"({active_subs/total_subs*100:.1f}%)")
print(f"Churned:              {churned_subs} "
      f"({churned_subs/total_subs*100:.1f}%)")
print(f"With upgrade:         {upgrades} "
      f"({upgrades/total_subs*100:.1f}%)")
print(f"With downgrade:       {downgrades} "
      f"({downgrades/total_subs*100:.1f}%)")

print(f"\nMRR range: "
      f"${subscriptions['mrr_amount'].min():,} → "
      f"${subscriptions['mrr_amount'].max():,}")
print(f"Avg MRR per subscription: "
      f"${subscriptions['mrr_amount'].mean():,.0f}")

# MRR by plan tier
print(f"\nAvg MRR by plan tier:")
print(subscriptions.groupby('plan_tier')['mrr_amount'].mean()
      .round(0).sort_values(ascending=False))

=== Subscription Summary ===
Total subscriptions:  5000
Active (no end_date): 4514 (90.3%)
Churned:              486 (9.7%)
With upgrade:         529 (10.6%)
With downgrade:       218 (4.4%)

MRR range: $0 → $33,830
Avg MRR per subscription: $2,268

Avg MRR by plan tier:
plan_tier
Enterprise   4918.00
Pro          1257.00
Basic         475.00
Name: mrr_amount, dtype: float64


## 6. Key Findings & Data Quality Notes

**Data quality:**
- All foreign keys valid — no orphaned records
- end_date nulls in subscriptions = active subscriptions (expected)
- satisfaction_score nulls in support = no customer response (expected)
- feedback_text nulls in churn = no written feedback (expected)

**Expansion universe defined:**
- Active accounts on Basic or Pro plan
- These are the candidates for upsell analysis in subsequent notebooks

**Notes for analysis:**
- 600 churn events for 500 accounts — reactivations (~10%) explain the excess
- Treat churn_flag in accounts as the source of truth for active/churned status
- Enterprise accounts excluded from expansion scoring (no higher tier)

In [15]:
# Save active accounts for use in subsequent notebooks
active_accounts.to_csv(
    '../data/processed/active_accounts.csv', index=False
)
expansion_universe.to_csv(
    '../data/processed/expansion_universe.csv', index=False
)

print("Saved:")
print(f"  data/processed/active_accounts.csv    "
      f"({len(active_accounts)} rows)")
print(f"  data/processed/expansion_universe.csv "
      f"({len(expansion_universe)} rows)")

Saved:
  data/processed/active_accounts.csv    (390 rows)
  data/processed/expansion_universe.csv (270 rows)
